# Matricula Online doorzoekbaar maken (OCR, zonder de scans te bewaren)

Dit notebook haalt de scans van een Matricula-boek één voor één op, laat er OCR op los **in het geheugen**, en bewaart alleen de tekst in een SQLite-database. Elke treffer verwijst terug naar de scan op Matricula (`…/?pg=N`), dus Matricula blijft de plek waar je het beeld bekijkt.

**Werkwijze:** cellen 1 t/m 3 eenmalig draaien, dan in cel 4 je boek-URL('s) invullen en draaien. Die verwerking loopt op de achtergrond, dus je kunt intussen al zoeken in cel 6. Daarna verbeter je de uitkomst met de stappen in cel 5.

**Wat je moet weten**
- Bedoeld voor **getypte** bronnen zoals de *Status Animarum/Abschrift* van Ahaus. Op handschrift (de originele kerkboeken) levert Tesseract onbruikbare tekst op; daarvoor heb je HTR nodig (bijv. Transkribus).
- Getest op `SAA_AhaM`: ruim 90% van de woorden komt goed door de OCR. Opgehoogde hoofdletters (ᴴenricus, ᴹaria) gaan vaak mis ('Penricus', 'karia'). Het fuzzy zoeken vangt een deel daarvan op, maar controleer altijd op de scan.
- De kolom *ruwe structuur* (relatie, leeftijd, geschat geboortejaar = beginjaar van het boek min leeftijd) is een **heuristische zoekhulp**, geen transcriptie.
- Het notebook wacht 3 seconden tussen scans. Laat dat zo; Matricula is een non-profitdienst.

## 1. Installeren

In [ ]:
!apt-get -qq install -y tesseract-ocr tesseract-ocr-lat tesseract-ocr-deu > /dev/null
!pip -q install pytesseract rapidfuzz
!tesseract --list-langs

## 2. Opslag
De database staat tijdens het werk in `/content` en wordt na elke verwerkte pagina naar Google Drive gekopieerd. Bij een vastloper ben je hooguit één pagina kwijt. Zet `GEBRUIK_DRIVE = False` als je dat niet wilt; download de database dan zelf via cel 7 voordat de sessie stopt.

In [ ]:
import os, shutil, sqlite3
GEBRUIK_DRIVE = True
DRIVE_MAP = "/content/drive/MyDrive/matricula_ocr"
DB_LOKAAL = "/content/matricula_ocr.db"

if GEBRUIK_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_MAP, exist_ok=True)
    DB_DRIVE = os.path.join(DRIVE_MAP, "matricula_ocr.db")
    if os.path.exists(DB_DRIVE) and not os.path.exists(DB_LOKAAL):
        shutil.copy(DB_DRIVE, DB_LOKAAL)
        print("Bestaande database van Drive geladen – je gaat verder waar je was.")

def bewaar_op_drive(conn, stil=False):
    # SQLite-backup (consistent, ook terwijl er gezocht wordt) naar een tijdelijk bestand,
    # dat daarna de vorige versie vervangt: een afgebroken kopie beschadigt nooit de laatste goede.
    if not GEBRUIK_DRIVE:
        return
    tmp = "/content/_kopie.db"
    doel = sqlite3.connect(tmp)
    conn.backup(doel)
    doel.close()
    shutil.copy(tmp, DB_DRIVE + ".tmp")
    os.replace(DB_DRIVE + ".tmp", DB_DRIVE)
    if not stil:
        print("→ database gekopieerd naar", DB_DRIVE)

## 3. De machinerie
Drie cellen: toegang tot Matricula, OCR + structuur, database + zoeken. Gewoon draaien.

In [ ]:
# ----- instellingen en toegang tot Matricula -----
import difflib, hashlib, io, re, sqlite3, time, json, html
from ast import literal_eval
from urllib.parse import urlsplit, urlunsplit

import requests
import numpy as np
import cv2
import pytesseract
from bs4 import BeautifulSoup
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

PAUZE_SEC = 3          # wachttijd tussen beeldverzoeken (wees vriendelijk voor Matricula)
OCR_TAAL = "lat+deu"   # getest: beter dan alleen 'lat' of 'eng+lat'
OCR_PSM = 4            # getest op SAA_AhaM: psm 4 iets beter dan 3 of 6

LOGBESTAND = "/content/voortgang.log"

def log(tekst):
    """Schrijft voortgang naar een logbestand (de verwerking draait op de achtergrond,
    dus print() zou in een willekeurige cel verschijnen)."""
    regel = time.strftime("%H:%M:%S ") + tekst
    try:
        with open(LOGBESTAND, "a", encoding="utf-8") as f:
            f.write(regel + "\n")
    except OSError:
        print(regel)

UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")


# Matricula ondertekent elke beeld-URL met een 'ctrl'-parameter.
# De scraper van 1fge doet dat via een naar Python vertaalde JavaScript-routine (js2py),
# maar js2py werkt niet meer op Python 3.12 (Colab). Die routine blijkt gewoon
# MD5 + AES-256 (ECB) met een vaste sleutel te zijn; hieronder dezelfde berekening in drie regels.
_SLEUTEL = bytes([112, 71, 53, 56, 38, 81, 106, 36, 63, 46, 100, 61, 40, 60, 46, 91,
                  118, 33, 107, 74, 107, 109, 95, 88, 78, 98, 115, 97, 54, 101, 46, 102])

def beeld_url(pad, csrf):
    b = f"{pad}?csrf={csrf}"
    digest = hashlib.md5(b.encode()).digest()
    enc = Cipher(algorithms.AES(_SLEUTEL), modes.ECB()).encryptor()
    ctrl = (enc.update(digest) + enc.finalize()).hex()
    return f"https://img.data.matricula-online.eu{b}&ctrl={ctrl}"

def basis_url(url):
    """Boek-URL zonder ?pg=..."""
    p = urlsplit(url)
    pad = p.path if p.path.endswith("/") else p.path + "/"
    return urlunsplit((p.scheme, p.netloc, pad, "", ""))

def lees_boek(url, sessie):
    """Haalt de boekpagina op: lijst van beeldpaden + labels + metadata."""
    r = sessie.get(basis_url(url), headers={"User-Agent": UA}, timeout=30)
    r.raise_for_status()
    csrf = sessie.cookies.get("shared_csrftoken")
    if not csrf:
        raise RuntimeError("Geen CSRF-token ontvangen; heeft Matricula iets veranderd?")
    tekst = r.text
    bestanden = literal_eval(tekst.split('"files":')[1].split('"],')[0].strip() + '"]')
    labels = literal_eval(tekst.split('"labels":')[1].split('"],')[0].strip() + '"]')

    soup = BeautifulSoup(tekst, "html.parser")
    meta = {}
    for tr in soup.select("table.table-register-data tr"):
        cellen = [c.get_text(" ", strip=True) for c in tr.find_all(["th", "td"])]
        if len(cellen) >= 2:
            meta[cellen[0]] = cellen[1]
    boek_id = meta.get("ID") or urlsplit(basis_url(url)).path.rstrip("/").split("/")[-1]
    jaar = re.search(r"\d{4}", meta.get("Date from", ""))
    return {
        "boek_id": boek_id,
        "titel": meta.get("Type", boek_id),
        "jaar": int(jaar.group()) if jaar else None,
        "basis_url": basis_url(url),
        "csrf": csrf,
        "paginas": list(zip(bestanden, labels)),
    }

def haal_beeld(pad, csrf, sessie):
    """Haalt één scan op als bytes – in het geheugen, niets op schijf."""
    h = {"User-Agent": UA, "Referer": "https://data.matricula-online.eu/",
         "Origin": "https://data.matricula-online.eu"}
    for poging in range(3):
        try:
            r = sessie.get(beeld_url(pad, csrf), headers=h, timeout=60)
            if r.status_code == 200 and r.headers.get("content-type", "").startswith("image"):
                return r.content
            log(f"   ! status {r.status_code}")
            return None
        except requests.RequestException as e:
            log(f"   ! netwerkfout ({e.__class__.__name__}), nieuwe poging…")
            time.sleep(5 * (poging + 1))
    return None

In [ ]:
# ----- OCR, opschonen en ruwe structuur -----
def ocr(beeld_bytes):
    """OCR op het grijswaardenbeeld. Getest: drempelen/binariseren tegen het
    doorschijnen maakte het resultaat op deze scans juist slechter, dus geen voorbewerking."""
    g = cv2.imdecode(np.frombuffer(beeld_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)
    d = pytesseract.image_to_data(g, lang=OCR_TAAL, config=f"--psm {OCR_PSM}",
                                  output_type=pytesseract.Output.DICT)
    # woorden -> regels (met verticale positie)
    regels = {}
    for i, w in enumerate(d["text"]):
        if not w.strip() or float(d["conf"][i]) < 0:
            continue
        k = (d["block_num"][i], d["par_num"][i], d["line_num"][i])
        r = regels.setdefault(k, {"woorden": [], "top": 10**9, "bodem": 0, "links": 10**9})
        r["woorden"].append(w)
        r["top"] = min(r["top"], d["top"][i])
        r["bodem"] = max(r["bodem"], d["top"][i] + d["height"][i])
        r["links"] = min(r["links"], d["left"][i])
    regels = sorted(regels.values(), key=lambda r: r["top"])
    return regels

def regels_naar_alineas(regels):
    """Splitst in alinea's (= huishoudens) op basis van witruimte tussen regels."""
    if not regels:
        return []
    hoogtes = sorted(r["bodem"] - r["top"] for r in regels)
    h = hoogtes[len(hoogtes) // 2] or 20
    alineas, huidig, vorige_bodem = [], [], None
    for r in regels:
        tekst = " ".join(r["woorden"])
        if is_kop(tekst):                 # folio- of letterkop: alinea afsluiten, kop weglaten
            if huidig:
                alineas.append(huidig); huidig = []
            vorige_bodem = None
            continue
        if vorige_bodem is not None and r["top"] - vorige_bodem > 0.55 * h:
            alineas.append(huidig); huidig = []
        huidig.append(tekst)
        vorige_bodem = r["bodem"]
    if huidig:
        alineas.append(huidig)
    return [plak_regels(a) for a in alineas]

def plak_regels(regels):
    """Plakt regels aan elkaar en herstelt afbrekingen ('Theo-' + 'dorus')."""
    uit = ""
    for r in regels:
        r = r.strip()
        if uit.endswith("-") and r[:1].islower():
            uit = uit[:-1] + r
        else:
            uit = (uit + " " + r).strip()
    return re.sub(r"\s+", " ", uit)

KOPREGEL = re.compile(r"^(f\.\s?\d+\S*|[A-Z]\.?|Hs\.\s?\d+.*|.{0,3})$")

def is_kop(tekst):
    """Folionummers, letterkoppen ('A', 'B') en ruis van één, twee tekens."""
    return bool(KOPREGEL.match(tekst.strip()))


RELATIES = {
    "uxor": "echtgenote", "maritus": "echtgenoot",
    "filius": "zoon", "filia": "dochter", "fil.": "kind", "fil": "kind", "filiolus": "zoontje", "filiola": "dochtertje",
    "pater": "vader", "mater": "moeder", "frater": "broer", "soror": "zus",
    "ancilla": "dienstmeid", "serva": "dienstmeid", "servus": "knecht", "famula": "dienstmeid",
    "famulus": "knecht", "bubulcus": "koeherder", "nepos": "kleinzoon/neef", "neptis": "kleindochter/nicht",
    "socrus": "schoonmoeder", "socer": "schoonvader", "privignus": "stiefzoon", "privigna": "stiefdochter",
    "vidua": "weduwe", "viduus": "weduwnaar", "inquilinus": "inwoner", "incola": "inwoner",
}
_rel_woorden = sorted(RELATIES, key=len, reverse=True)
_rel_re = re.compile(r"^(" + "|".join(re.escape(k) for k in _rel_woorden) + r")(?=[\s.]|$)\.?\s*", re.I)
# relatiewoord midden in een stuk zonder komma ervoor ('Adriana 4 ancilla Elisabetha')
_rel_midden = re.compile(r"(?<=\d)\s+(?=(" + "|".join(re.escape(k) for k in _rel_woorden) + r")\b)", re.I)
_INFANS = r"i[nr]f[a-z]{2}s"   # infans, ook OCR-varianten als 'irfans', 'infaus'
_leeftijd_re = re.compile(r"(?:^|\s)((?=[0-9COlI]*\d)[0-9COlI]{1,3}|ll|" + _INFANS + r")(?:\s+\S{1,2})?\s*[.,;:]?\s*$", re.I)
_CIJFERFIX = str.maketrans({"C": "0", "O": "0", "l": "1", "I": "1"})

def _leeftijd(deel):
    m = _leeftijd_re.search(deel)
    if not m:
        return None, deel
    tok = m.group(1)
    if re.fullmatch(_INFANS, tok, re.I):
        return 0, deel[:m.start()].strip()
    try:
        n = int(tok.translate(_CIJFERFIX))
    except ValueError:
        return None, deel
    return (n if n <= 110 else None), deel[:m.start()].strip()

def _relatie_van(woord):
    """Relatiewoord herkennen, ook met een OCR-fout ('umor' -> uxor)."""
    w = woord.lower().strip(".:")
    if w in RELATIES:
        return RELATIES[w]
    if len(w) >= 4:
        kand = difflib.get_close_matches(w, [k for k in RELATIES if len(k) >= 4], n=1, cutoff=0.75)
        if kand:
            return RELATIES[kand[0]]
    return None

def ontleed(alinea, refjaar):
    """Heuristische opsplitsing: hoofd + leden met leeftijd en geschat geboortejaar.
    Bedoeld als zoekhulp, niet als transcriptie: altijd tegen de scan controleren."""
    alinea = re.sub(r"\bfil\.(?=\S)", "fil. ", alinea)
    stukken = []
    for p in alinea.split(","):
        stukken += [x.strip() for x in _rel_midden.split(p) if x and not _rel_re.fullmatch(x.strip() + " ")]
    personen, laatste_rel = [], "hoofd"
    for deel in stukken:
        if not deel:
            continue
        eerste, _, rest = deel.partition(" ")
        rel = _relatie_van(eerste)
        if rel:
            deel = re.sub(r"^(major|minor|ejus|eius)\b\s*", "", rest, flags=re.I)
        elif personen and personen[-1]["leeftijd"] is None:
            # geen relatiewoord en vorige persoon nog zonder leeftijd: zelfde persoon
            # ('Bernardus Almerinck, faber lignarius 35')
            vorige = personen[-1]
            leeftijd, rest = _leeftijd(deel)
            vorige["tekst"] = (vorige["tekst"] + ", " + rest).strip(", ")
            vorige["leeftijd"] = leeftijd
            continue
        else:
            rel = laatste_rel if personen else "hoofd"   # 'filia Adelheidis 10, Helena 4'
        laatste_rel = rel
        leeftijd, deel = _leeftijd(deel)
        personen.append({"relatie": rel, "tekst": deel, "leeftijd": leeftijd})
    for p in personen:
        p["geb_ca"] = (refjaar - p["leeftijd"]) if (refjaar and p["leeftijd"] is not None) else None
    return personen

In [ ]:
# ----- database en zoeken -----
def open_db(pad):
    db = sqlite3.connect(pad, timeout=60)
    db.execute("PRAGMA journal_mode=WAL")   # lezen (zoeken) terwijl er geschreven wordt
    db.executescript("""
    CREATE TABLE IF NOT EXISTS boeken(boek_id TEXT PRIMARY KEY, titel TEXT, jaar INT, basis_url TEXT);
    CREATE TABLE IF NOT EXISTS paginas(boek_id TEXT, pg INT, label TEXT, ruwe_tekst TEXT,
                                       PRIMARY KEY(boek_id, pg));
    CREATE TABLE IF NOT EXISTS huishoudens(id INTEGER PRIMARY KEY, boek_id TEXT, pg INT, nr INT, tekst TEXT);
    CREATE TABLE IF NOT EXISTS personen(huishouden_id INT, relatie TEXT, tekst TEXT, leeftijd INT, geb_ca INT);
    CREATE VIRTUAL TABLE IF NOT EXISTS zoek USING fts5(tekst, huishouden_id UNINDEXED, tokenize='trigram');
    """)
    return db

def verwerk_boek(url, db, max_paginas=None, na_pagina=None, stop=None):
    """na_pagina: functie die na elke voltooide pagina wordt aangeroepen (kopie naar Drive).
    stop: threading.Event; als die gezet wordt, stopt de verwerking na de lopende pagina."""
    sessie = requests.Session()
    boek = lees_boek(url, sessie)
    bid = boek["boek_id"]
    db.execute("INSERT OR REPLACE INTO boeken VALUES (?,?,?,?)",
               (bid, boek["titel"], boek["jaar"], boek["basis_url"]))
    klaar = {r[0] for r in db.execute("SELECT pg FROM paginas WHERE boek_id=?", (bid,))}
    paginas = boek["paginas"][:max_paginas] if max_paginas else boek["paginas"]
    log(f"{bid}: {boek['titel']} ({boek['jaar']}) – {len(boek['paginas'])} scans, "
        f"{len(klaar)} al gedaan")
    for idx, (pad, label) in enumerate(paginas):
        pg = idx + 1                      # Matricula telt ?pg= vanaf 1 (AS_000 = pg 1)
        if pg in klaar:
            continue
        if stop is not None and stop.is_set():
            log(f"{bid}: gestopt op verzoek")
            return bid
        beeld = haal_beeld(pad, boek["csrf"], sessie)
        time.sleep(PAUZE_SEC)
        if beeld is None:
            log(f" · {bid} {label} (pg={pg}) – overgeslagen"); continue
        regels = ocr(beeld)
        del beeld                         # beeld weg; alleen tekst blijft
        ruw = "\n".join(" ".join(r["woorden"]) for r in regels)
        db.execute("INSERT OR REPLACE INTO paginas VALUES (?,?,?,?)", (bid, pg, label, ruw))
        oud = [r[0] for r in db.execute("SELECT id FROM huishoudens WHERE boek_id=? AND pg=?", (bid, pg))]
        for hid in oud:
            db.execute("DELETE FROM zoek WHERE huishouden_id=?", (hid,))
            db.execute("DELETE FROM personen WHERE huishouden_id=?", (hid,))
        db.execute("DELETE FROM huishoudens WHERE boek_id=? AND pg=?", (bid, pg))
        n = 0
        for nr, alinea in enumerate(regels_naar_alineas(regels)):
            if is_kop(alinea):
                continue
            cur = db.execute("INSERT INTO huishoudens(boek_id,pg,nr,tekst) VALUES (?,?,?,?)",
                             (bid, pg, nr, alinea))
            hid = cur.lastrowid
            db.execute("INSERT INTO zoek(tekst, huishouden_id) VALUES (?,?)", (alinea, hid))
            for p in ontleed(alinea, boek["jaar"]):
                db.execute("INSERT INTO personen(huishouden_id, relatie, tekst, leeftijd, geb_ca) VALUES (?,?,?,?,?)",
                           (hid, p["relatie"], p["tekst"], p["leeftijd"], p["geb_ca"]))
            n += 1
        db.commit()
        log(f" · {bid} {label} (pg={pg}) – {n} huishoudens")
        if na_pagina:
            na_pagina()
    return bid


import unicodedata
from rapidfuzz import fuzz

def norm(w):
    """Grove klanknormalisatie voor Westfaalse/Latijnse namen:
    Büninck = Bünink = Buning, Ahlmerinck = Almerinck, Theodorus = Teodorus."""
    w = unicodedata.normalize("NFKD", w.lower())
    w = "".join(c for c in w if c.isalpha())
    for a, b in [("ck", "k"), ("c", "k"), ("th", "t"), ("ph", "f"), ("y", "i"), ("j", "i"),
                 ("v", "f"), ("w", "f"), ("z", "s")]:
        w = w.replace(a, b)
    w = re.sub(r"(?<=[aeiou])h(?![aeiou])", "", w)
    w = re.sub(r"(.)\1+", r"\1", w)
    w = re.sub(r"(ink|ing|ik)$", "ink", w)
    return w

def _bewerkt(db):
    return "tekst_ocr" in [r[1] for r in db.execute("PRAGMA table_info(huishoudens)")]

def zoek(db, term, fuzzy=True, drempel=85, geb_van=None, geb_tot=None, max_res=200):
    """Zoekt huishoudens, in de verbeterde tekst én in de oorspronkelijke OCR."""
    term = term.strip()
    if not term:
        return []
    ocr_kol = "tekst_ocr" if _bewerkt(db) else "tekst"
    if fuzzy:
        tw = [norm(t) for t in term.split() if norm(t)]
        uit = []
        for hid, bid, pg, tekst, ocr in db.execute(f"SELECT id, boek_id, pg, tekst, {ocr_kol} FROM huishoudens"):
            woorden = {norm(w) for w in re.findall(r"[^\W\d_]+", (tekst or "") + " " + (ocr or ""))} - {""}
            if not woorden:
                continue
            s = min(max(fuzz.ratio(t, w) for w in woorden) for t in tw)
            if s >= drempel:
                uit.append((s, hid, bid, pg, tekst))
        uit.sort(key=lambda r: -r[0])
    else:
        if len(term) < 3:
            return []
        q = '"' + term.replace('"', '""') + '"'
        uit = [(100, hid, bid, pg, tekst) for hid, bid, pg, tekst in db.execute(
            """SELECT h.id, h.boek_id, h.pg, h.tekst FROM zoek z JOIN huishoudens h
               ON h.id = z.huishouden_id WHERE zoek MATCH ?""", (q,))]
    if geb_van or geb_tot:
        lo, hi = geb_van or 0, geb_tot or 9999
        ok = {r[0] for r in db.execute(
            "SELECT DISTINCT huishouden_id FROM personen WHERE geb_ca BETWEEN ? AND ?", (lo, hi))}
        uit = [r for r in uit if r[1] in ok]
    return uit[:max_res]

def toon(db, resultaten, term=""):
    """Resultaten als HTML-tabel; verbeterde woorden onderstreept (aanwijzen = wat de OCR las)."""
    bewerkt = _bewerkt(db)
    boeken = {b: (t, u) for b, t, u in db.execute("SELECT boek_id, titel, basis_url FROM boeken")}
    labels = {(b, p): l for b, p, l in db.execute("SELECT boek_id, pg, label FROM paginas")}
    tw = [norm(t) for t in term.split()]
    def markeer(tekst, corr):
        uit = []
        for i, tok in enumerate((tekst or "").split(" ")):
            e = html.escape(tok)
            if tw and any(max(fuzz.ratio(norm(w), t) for t in tw) >= 80 for w in re.findall(r"[^\W\d_]+", tok)):
                e = f"<mark>{e}</mark>"
            if i in corr:
                e = f"<u title='OCR: {html.escape(corr[i][0])} ({html.escape(corr[i][1])})'>{e}</u>"
            uit.append(e)
        return " ".join(uit)
    rijen = []
    for s, hid, bid, pg, tekst in resultaten:
        titel, url = boeken.get(bid, (bid, ""))
        corr, extra = {}, ""
        if bewerkt:
            sleutel, ocr, reden = db.execute("SELECT sleutel, tekst_ocr, begin_reden FROM huishoudens WHERE id=?",
                                             (hid,)).fetchone()
            for pos, o, laag in db.execute("SELECT positie, ocr, laag FROM correcties WHERE huishouden_id=? "
                                           "AND laag != 'rommelteken' AND positie IS NOT NULL", (hid,)):
                corr[pos] = (o, laag)
            extra = (f"<details><summary><small>oorspronkelijke OCR · {html.escape(sleutel or '')}</small></summary>"
                     f"<small>{html.escape(ocr or '')}<br><i>huishouden begint: {html.escape(reden or '')}</i>"
                     f"</small></details>")
        pers = db.execute("SELECT relatie, tekst, leeftijd, geb_ca FROM personen WHERE huishouden_id=?",
                          (hid,)).fetchall()
        pers_html = "<br>".join(
            f"<small>{html.escape(r or '')}: {html.escape(t or '–')}"
            f"{f' ({l}, ca. {g})' if l is not None and g else ''}</small>" for r, t, l, g in pers)
        rijen.append(
            f"<tr><td style='white-space:nowrap'><a href='{url}?pg={pg}' target='_blank'>"
            f"{html.escape(bid)} · {html.escape(labels.get((bid, pg), str(pg)))}</a><br><small>score {s:.0f}</small></td>"
            f"<td>{markeer(tekst, corr)}{extra}</td><td>{pers_html}</td></tr>")
    kop = (f"<p><b>{len(resultaten)}</b> huishoudens gevonden</p>"
           "<table style='border-collapse:collapse' border=1 cellpadding=5>"
           "<tr><th>Scan</th><th>Tekst (onderstreept = automatisch verbeterd)</th><th>Ruwe structuur (controleren!)</th></tr>")
    return kop + "".join(rijen) + "</table>"

db = open_db(DB_LOKAAL)
print('Klaar. Pagina\'s in database:', db.execute('SELECT COUNT(*) FROM paginas').fetchone()[0])

### 3b. (optioneel) Eerst één scan proberen
Toont één scan met de OCR-uitkomst ernaast, zonder iets op te slaan. Handig om te zien of een boek geschikt is voor OCR voordat je het hele boek laat lopen.

In [ ]:
from IPython.display import display, HTML
import base64
PROBEER_URL = "https://data.matricula-online.eu/en/deutschland/muenster/0-status-animarum/SAA_AhaM/?pg=2"

_s = requests.Session()
_boek = lees_boek(PROBEER_URL, _s)
_pg = int(re.search(r"pg=(\d+)", PROBEER_URL).group(1)) if "pg=" in PROBEER_URL else 1
_pad, _label = _boek["paginas"][_pg - 1]
_img = haal_beeld(_pad, _boek["csrf"], _s)
_alineas = [a for a in regels_naar_alineas(ocr(_img)) if not is_kop(a)]
display(HTML(
    f"<h4>{html.escape(_boek['titel'])} · {_label} · {len(_boek['paginas'])} scans in dit boek</h4>"
    "<div style='display:flex;gap:16px'>"
    f"<img src='data:image/jpeg;base64,{base64.b64encode(_img).decode()}' style='width:48%'>"
    "<ol style='width:48%;font-family:monospace;font-size:12px'>"
    + "".join(f"<li>{html.escape(a)}</li>" for a in _alineas) + "</ol></div>"))
del _img

## 4. Boeken verwerken (op de achtergrond)
Zet één of meer boek-URL's in de lijst (welke `?pg=` erachter staat maakt niet uit) en draai de cel. De verwerking start **op de achtergrond**: de cel is meteen klaar en je kunt intussen zoeken in cel 6. Alles wat al verwerkt is, is direct doorzoekbaar.

Pagina's die al in de database staan worden overgeslagen, dus na een onderbreking draai je deze cel gewoon opnieuw. Reken op ruwweg 6–8 seconden per scan.

In [ ]:
import threading

BOEKEN = [
    "https://data.matricula-online.eu/en/deutschland/muenster/0-status-animarum/SAA_AhaM/?pg=2",
]
MAX_PAGINAS = None    # bijv. 3 om eerst te testen

def _verwerk_alles(boeken, stop):
    conn = open_db(DB_LOKAAL)          # eigen verbinding voor deze achtergrondtaak
    try:
        for url in boeken:
            if stop.is_set():
                break
            try:
                verwerk_boek(url, conn, max_paginas=MAX_PAGINAS, stop=stop,
                             na_pagina=lambda: bewaar_op_drive(conn, stil=True))
            except Exception as e:
                log(f"FOUT bij {url}: {e!r}")
    finally:
        bewaar_op_drive(conn, stil=True)
        conn.close()
        log("=== GESTOPT ===" if stop.is_set() else "=== ALLES KLAAR ===")

if "werker" in globals() and werker.is_alive():
    print("Er loopt al een verwerking. Stop die eerst met cel 4c, of wacht tot hij klaar is.")
else:
    STOP = threading.Event()
    werker = threading.Thread(target=_verwerk_alles, args=(list(BOEKEN), STOP), daemon=True)
    werker.start()
    print(f"Gestart op de achtergrond met {len(BOEKEN)} boek(en). Voortgang: cel 4b. Zoeken: cel 6.")

### 4b. Voortgang bekijken
Draai deze cel zo vaak als je wilt.

In [ ]:
_nodig = [n for n in ("open_db", "lees_boek", "haal_beeld", "log", "bewaar_op_drive", "DB_LOKAAL") if n not in globals()]
if _nodig:
    raise RuntimeError("Draai eerst cel 1 tot en met 3: installeren, opslag (Drive) en de drie cellen onder "
                       "'De machinerie'. Na een herstart van Colab moet dat opnieuw. Ontbreekt nu: " + ", ".join(_nodig))
def voortgang(regels=15):
    actief = "werker" in globals() and werker.is_alive()
    print("Verwerking:", "LOOPT" if actief else "staat stil")
    for bid, titel, n in db.execute("""SELECT b.boek_id, b.titel, COUNT(p.pg) FROM boeken b
                                      LEFT JOIN paginas p USING(boek_id) GROUP BY b.boek_id"""):
        print(f"  {bid:14} {n:4} pagina's  {titel}")
    if os.path.exists(LOGBESTAND):
        print("\nLaatste meldingen:")
        print("".join(open(LOGBESTAND, encoding="utf-8").readlines()[-regels:]))

voortgang()

### 4c. Stoppen
Stopt netjes na de pagina waar hij mee bezig is en kopieert de database naar Drive. Later verdergaan = cel 4 opnieuw draaien.

In [ ]:
STOP.set()
werker.join(timeout=120)
print("Gestopt." if not werker.is_alive() else "Nog bezig met de laatste pagina; kijk zo in cel 4b.")

## 5. Verbeteren (stap 1–3)
Drie bewerkingen die de OCR-uitkomst verbeteren. **De oorspronkelijke OCR wordt nooit overschreven**; elke automatische wijziging is in de database terug te vinden (tabellen `bewerkingen`, `regels`, `correcties`, en per huishouden `begin_reden`).

- **Stap 3** (eerst, want duurt lang): de boeken met één persoon per regel worden opnieuw opgehaald en **per kolom** geOCR'd (rol · naam · leeftijd · opmerking). Ongeveer 730 scans. Reken op ongeveer een uur tot anderhalf uur: downloaden en lezen lopen nu door elkaar heen en de kolommen worden tegelijk gelezen. Stopt de Colab-sessie eerder, dan draai je 5c gewoon opnieuw. Loopt op de achtergrond en is na een onderbreking te hervatten.
- **Stap 1 en 2** (daarna, duurt een halve minuut): huishoudens opnieuw splitsen en de OCR-correctie. Dit bouwt huishoudens, personen, correcties en de zoekindex opnieuw op uit de ruwe OCR, dus je kunt het zo vaak draaien als je wilt.

Draai cel 5a en 5b eenmalig (ze schrijven de code naar bestanden), daarna 5c voor stap 3 en, als die klaar is, 5e voor stap 1 en 2.

In [ ]:
%%writefile /content/bewerk.py
# bewerk.py – stap 1 (huishoudens opnieuw splitsen) en stap 2 (OCR-correctie)
# voor de database van het Status-Animarum-notebook.
#
# Uitgangspunten:
#  * paginas.ruwe_tekst (de OCR) wordt nooit gewijzigd;
#  * elke automatische wijziging is terug te voeren op een regel en een versie
#    (tabellen bewerkingen, regels, correcties);
#  * het script is herhaalbaar: opnieuw draaien bouwt huishoudens, personen,
#    correcties en zoekindex opnieuw op uit de ruwe OCR.

import collections, datetime, difflib, json, re, sqlite3, statistics, unicodedata
from rapidfuzz import process, distance

VERSIE = "2026-11-04.1"

# ---------------------------------------------------------------- algemeen
WOORD = r"[A-Za-zÄÖÜäöüßéèáàëïæ]+"

def plak_regels(regels):
    """Regels aan elkaar plakken en afbrekingen herstellen ('Theo-' + 'dorus')."""
    uit = ""
    for r in regels:
        r = r.strip()
        if uit.endswith("-") and r[:1].islower():
            uit = uit[:-1] + r
        else:
            uit = (uit + " " + r).strip()
    return re.sub(r"\s+", " ", uit)

# ---------------------------------------------------------------- stap 1: splitsen
KOP_OUD = re.compile(r"^(f\.\s?\d+\S*|[A-Z]\.?|Hs\.\s?\d+.*|.{0,3})$")          # regel uit notebook v1
KOP = re.compile(r"^([f*£t]\s?\.\s?\d+\S*|[A-Z]\.?|Hs\.\s?\d+.*|.{0,3}|[^A-Za-zÄÖÜäöüß]*)$")
MARGE = re.compile(r"^\s*(f\.\s?\d+['’]?\s+|\d{1,3}\.\s+)")
KOPWOORD = re.compile(
    r"^(colonus|coloni|vidua|viduus|vid|kötter|kotter|pferdekötter|brinksitzer|brincksitzer|"
    r"mercenarius|mercenaria|mercen|villicus|incola|incolae|inquilinus|inquilini|inquil|textor|zeller|"
    r"schultetus|casatus|casat|par|familia|herr|operarius|operaria|heuerling|heuerman|arbeitsman|"
    r"einwohner|einwohnerin|tagelöhner|taglöhner)\b\.?", re.I)

def is_kop(l):
    return bool(KOP.match(l.strip()))

def oude_bereiken(lines, hh_teksten):
    """Koppelt de huishoudens uit het notebook (gesplitst op witregels) terug aan regelnummers."""
    idx = [k for k, l in enumerate(lines) if l.strip() and not KOP_OUD.match(l.strip())]
    res, p = [], 0
    for t in hh_teksten:
        for q in range(p + 1, len(idx) + 1):
            if plak_regels([lines[k] for k in idx[p:q]]) == t:
                res.append((idx[p], idx[q - 1])); p = q; break
        else:
            return None
    return res

def schoon_einde(l):
    """Rommel aan het regeleinde weghalen (':', '||!', 'a RUE') vóór de eindcontrole."""
    l = MARGE.sub("", l).rstrip()
    vorige = None
    while vorige != l:
        vorige = l
        m = re.search(r"\s+(\S+)$", l)
        if m:
            tok = m.group(1)
            if not re.fullmatch(r"(\d{1,3}([/.]\d)?[.,;]?|\d/\d|[A-Za-zÄÖÜäöüß]{2,}[.,;:]?|[A-Za-zÄÖÜäöüß]+-)", tok) \
               or (len(tok) <= 3 and not tok[:1].isdigit()
                   and tok.lower().strip(".,") not in ("ux", "fil", "vid", "anc", "fam", "et")):
                l = l[:m.start()].rstrip()
    return l

def eind_signaal(l):
    l = l.rstrip(" |!'\"")
    if not l or l.endswith("-"):
        return "afbreking"
    if l.endswith(","):
        return "komma"
    if re.search(r"(\d|infans|[)\]])[.;]?$", l):
        return "getal"
    return "anders"

def grenzen(lines, witregel_grenzen, lijstboek, factor=0.82):
    """{regelnr: reden} – na deze regel eindigt een huishouden."""
    g = {a: "witregel" for a in witregel_grenzen}
    inhoud = [k for k, l in enumerate(lines) if l.strip() and not is_kop(l)]
    if lijstboek or len(inhoud) < 4:
        return g                     # lijstboeken: één persoon per regel, zie stap 3
    schoon = {k: schoon_einde(lines[k]) for k in inhoud}
    lens = sorted(len(schoon[k]) for k in inhoud)
    W = lens[int(0.85 * (len(lens) - 1))]
    for a, b in zip(inhoud, inhoud[1:]):
        if a in g:
            continue
        sig = eind_signaal(schoon[a])
        if sig in ("komma", "afbreking"):
            continue
        volgende = MARGE.sub("", lines[b]).lstrip(' |*"\'')
        m = KOPWOORD.match(volgende)
        if m and sig in ("getal", "anders"):
            g[a] = "volgende regel begint met '" + m.group(1).lower() + "'"
        elif sig == "getal" and len(schoon[a]) < factor * W:
            g[a] = "korte regel eindigend op een leeftijd"
    return g

def splits_pagina(lines, g):
    """Geeft [(regel_van, regel_tot, begin_reden, tekst)]."""
    uit, huidig, reden = [], [], "begin van de pagina"
    for k, l in enumerate(lines):
        if not l.strip() or is_kop(l):
            continue
        huidig.append(k)
        if k in g:
            uit.append((huidig[0], huidig[-1], reden, plak_regels([lines[i] for i in huidig])))
            huidig, reden = [], g[k]
    if huidig:
        uit.append((huidig[0], huidig[-1], reden, plak_regels([lines[i] for i in huidig])))
    return [u for u in uit if u[3].strip()]

# ---------------------------------------------------------------- stap 2: correctie
VERWARD_BEGIN = {
    "M": ["l", "L", "h", "k", "K", "N", "li", "Li", "ll", "lI", "Il"],
    "B": ["P", "D", "d", "S", "8"],
    "H": ["E", "h", "I"],
    "G": ["V", "C", "6"],
    "C": ["V", "G"],
    "E": ["L", "B"],
    "T": ["I", "l"],
    "J": ["d", "I", "l"],
}
# vormen die in dit corpus alleen als OCR-fout voorkomen, maar vaak genoeg om als 'naam' door te gaan
HANDMATIG = {"Meria": "Maria", "Naria": "Maria", "Neria": "Maria", "leria": "Maria", "lieria": "Maria",
             "eria": "Maria", "Mergaretha": "Margaretha", "Nargaretha": "Margaretha"}
VERV = {("s","a"),("g","a"),("z","a"),("r","n"),("n","r"),("e","c"),("c","e"),("i","l"),("l","i"),
        ("i","t"),("t","i"),("k","h"),("b","h"),("t","r"),("ä","d"),("ü","d"),("ü","u"),("ä","a"),
        ("m","n"),("h","n"),("e","s"),("o","a"),("t","b"),("p","o"),("T","F"),("c","d"),("v","c"),
        ("ö","o"),("á","a"),("é","e"),("I","l"),("l","I")}
RUIS = re.compile(r"^[^\wäöüÄÖÜß&/()'.,;:-]+$")
RAND = re.compile(r"^[|^»«*$#_~=<>{}\[\]\\\"!]+|[|^»«*$#_~=<>{}\[\]\\\"!]+$")
CIJFER = re.compile(r"^(\d)([CO])$|^([CO])(\d)$|^(ll)$")

class Corrector:
    def __init__(self, ruwe_teksten):
        self.freq = collections.Counter(); nb = collections.defaultdict(set)
        for boek, t in ruwe_teksten:
            for w in re.findall(WOORD, t or ""):
                self.freq[w] += 1; nb[w].add(boek)
        self.regels = self._bouw_regels()
        self.lexicon = {w for w, c in self.freq.items() if c >= 30 and len(nb[w]) >= 5
                        and len(w) >= 3 and w not in self.regels}
        self.lexlijst = sorted(self.lexicon)
        self.cache = {}

    def _bouw_regels(self, n_namen=200, min_freq=150):
        f = self.freq
        namen = [w for w, c in f.most_common() if w[:1].isupper() and len(w) >= 4
                 and w[1:].islower() and c >= min_freq and w not in HANDMATIG][:n_namen]
        echt = set(namen); regels = {}
        for n in namen:
            kand = {n[1:]: "beginletter weggevallen"}
            for ocr in VERWARD_BEGIN.get(n[0], []):
                kand[ocr + n[1:]] = f"{n[0]} gelezen als {ocr}"
            kand[n.lower()] = "hoofdletter gelezen als kleine letter"
            for v, reden in kand.items():
                if v in echt or len(v) < 3:
                    continue
                if f.get(v, 0) and f[v] * 3 <= f[n]:
                    regels[v] = (n, reden)
        for v, n in HANDMATIG.items():
            regels[v] = (n, "handmatig: komt in dit corpus alleen als OCR-fout voor")
        return regels

    @staticmethod
    def _verdacht(w, i):
        c = w[i]
        return (i > 0 and c.isupper()) or not c.isalpha() or c in "ßæ" or \
               (i > 0 and w[i-1].lower() == c.lower()) or (i + 1 < len(w) and w[i+1].lower() == c.lower())

    def _toegestaan(self, w, doel):
        if len(doel) < 4 or w.lower() == doel.lower():
            return False
        ops = distance.Levenshtein.editops(w, doel)
        if len(ops) != 1:
            return False
        op = ops[0]
        if op.tag == "replace":
            return len(w) >= 5 and ((w[op.src_pos], doel[op.dest_pos]) in VERV or self._verdacht(w, op.src_pos))
        if op.tag == "delete":
            return op.src_pos > 0 and len(w) >= 6 and self._verdacht(w, op.src_pos)
        if op.tag == "insert":
            return op.dest_pos == 0 and doel[0].isupper() and w[0].islower() and len(doel) >= 6
        return False

    def woord(self, w):
        """(nieuw, laag, regel) of None."""
        if w in self.cache:
            return self.cache[w]
        res = None
        if w in self.regels:
            res = (self.regels[w][0], "naamregel", w)
        elif len(w) >= 4 and self.freq.get(w, 0) <= 3 and w not in self.lexicon:
            beste = None
            for doel, dist, _ in process.extract(w, self.lexlijst, scorer=distance.Levenshtein.distance,
                                                 score_cutoff=1, limit=8):
                if self.freq[doel] >= 20 * max(self.freq.get(w, 1), 1) and self._toegestaan(w, doel):
                    if beste is None or self.freq[doel] > self.freq[beste]:
                        beste = doel
            if beste:
                res = (beste, "één teken", None)
        self.cache[w] = res
        return res

    def tekst(self, t):
        """Verbetert een huishoudtekst. Geeft (nieuwe tekst, [(positie_ocr, positie_nieuw, ocr, nieuw, laag, regel)]).
        positie_ocr = woordnummer in tekst_ocr, positie_nieuw = woordnummer in de verbeterde tekst (None als weg)."""
        uit, wijz = [], []
        for pos, tok in enumerate(t.split(" ")):
            if not tok:
                continue
            nu = len(uit)
            if RUIS.match(tok):
                wijz.append((pos, None, tok, "", "rommelteken", None)); continue
            kern = RAND.sub("", tok)
            if kern != tok:
                if not kern:
                    wijz.append((pos, None, tok, "", "rommelteken", None)); continue
                wijz.append((pos, nu, tok, kern, "rommelteken", None)); tok = kern
            if CIJFER.match(tok):
                n = tok.replace("C", "0").replace("O", "0").replace("ll", "11")
                wijz.append((pos, nu, tok, n, "cijfer", None)); uit.append(n); continue
            def vervang(m):
                r = self.woord(m.group(0))
                if r and r[0] != m.group(0):
                    wijz.append((pos, nu, m.group(0), r[0], r[1], r[2])); return r[0]
                return m.group(0)
            uit.append(re.sub(WOORD, vervang, tok))
        return " ".join(uit), wijz

# ---------------------------------------------------------------- ruwe structuur
RELATIES = {
    # Latijn, voluit en afgekort
    "uxor": "echtgenote", "ux": "echtgenote", "conj": "echtgenote", "conjux": "echtgenote",
    "maritus": "echtgenoot", "filius": "zoon", "filia": "dochter", "fil": "kind", "filii": "kinderen",
    "filiae": "dochters", "filiolus": "zoontje", "filiola": "dochtertje", "prol": "kind", "proles": "kind",
    "lib": "kind", "liberi": "kind", "pater": "vader", "mater": "moeder", "frater": "broer", "soror": "zus",
    "avus": "grootvader", "avia": "grootmoeder", "nepos": "kleinzoon/neef", "neptis": "kleindochter/nicht",
    "socrus": "schoonmoeder", "socer": "schoonvader", "gener": "schoonzoon", "nurus": "schoondochter",
    "privignus": "stiefzoon", "privigna": "stiefdochter", "ancilla": "dienstmeid", "ancillae": "dienstmeid",
    "anc": "dienstmeid", "serva": "dienstmeid", "servus": "knecht", "servi": "knecht", "serv": "knecht",
    "famulus": "knecht", "famula": "dienstmeid", "famuli": "dienstpersoneel", "fam": "dienstpersoneel",
    "bubulcus": "koeherder", "pastor": "herder", "vidua": "weduwe", "viduus": "weduwnaar", "vid": "weduwe",
    "incola": "inwoner", "incolae": "inwoners", "inc": "inwoner", "inquilinus": "inwoner", "inquil": "inwoner",
    # Duits
    "ehefrau": "echtgenote", "frau": "echtgenote", "kinder": "kind", "kind": "kind", "kindt": "kind",
    "sohn": "zoon", "tochter": "dochter", "vatter": "vader", "vater": "vader", "mutter": "moeder",
    "bruder": "broer", "schwester": "zus", "knecht": "knecht", "knegt": "knecht", "knegte": "knecht",
    "magd": "dienstmeid", "mohne": "tante", "wittibe": "weduwe", "witwe": "weduwe",
    "einwohner": "inwoner", "einwohnerin": "inwoonster", "ehel": "echtpaar",
}
_REL_TOK = re.compile(r"^(" + "|".join(sorted(map(re.escape, RELATIES), key=len, reverse=True)) + r")\.?:?$", re.I)
_LEEFTIJD = re.compile(r"^(\d{1,3}|\d?\s?1/[24]|3/4|[13]/4|1/2|infans)[.,;:]?$", re.I)

def _leeftijd_waarde(tok):
    t = tok.strip(".,;:").lower()
    if t == "infans" or "/" in t:
        return 0
    try:
        n = int(t)
        return n if n <= 110 else None
    except ValueError:
        return None

def ontleed(tekst, refjaar):
    """Heuristische opsplitsing van één huishouden in personen (zoekhulp, geen transcriptie)."""
    tekst = re.sub(r"\b(ux|fil|prol|lib|conj|vid|anc|fam|inc|serv|inquil)\.(?=\S)", r"\1. ", tekst, flags=re.I)
    tekst = re.sub(r"([A-Za-zäöüß]\.)(\d)", r"\1 \2", tekst)          # 'col.60' -> 'col. 60'
    personen, overerf = [], "hoofd"
    for i, seg in enumerate(s.strip() for s in tekst.split(",")):
        if not seg:
            continue
        toks = seg.split()
        rel = None; naam = []; leeftijd = None
        for j, t in enumerate(toks):
            m = _REL_TOK.match(t)
            if m and rel is None:
                rel = RELATIES[m.group(1).lower()]
            elif rel is None and j == 0 and len(t.strip(".:;,")) >= 5 and (
                    k := difflib.get_close_matches(t.strip(".:;,").lower(),
                                                   [x for x in RELATIES if len(x) >= 5], n=1, cutoff=0.75)):
                rel = RELATIES[k[0]]                 # OCR-variant van een rolwoord ('Shefrau', 'Kinat')
            elif _LEEFTIJD.match(t):
                if leeftijd is None:                     # eerste getal = leeftijd; later losse cijfers zijn meestal ruis
                    leeftijd = _leeftijd_waarde(t)
            else:
                naam.append(t)
        tekst_p = " ".join(naam).strip(" :")
        if rel is None and personen and personen[-1]["leeftijd"] is None and i > 0:
            vorige = personen[-1]                       # 'Bernardus Almerinck, faber lignarius 35'
            vorige["tekst"] = (vorige["tekst"] + ", " + tekst_p).strip(", ")
            vorige["leeftijd"] = leeftijd
            continue
        if rel is None:
            rel = (overerf if overerf != "hoofd" else "huisgenoot (relatie niet vermeld)") if personen else "hoofd"
        else:
            overerf = rel
        personen.append({"relatie": rel, "tekst": tekst_p, "leeftijd": leeftijd})
    for p in personen:
        p["geb_ca"] = (refjaar - p["leeftijd"]) if (refjaar and p["leeftijd"] is not None) else None
    return personen

# ---------------------------------------------------------------- database
SCHEMA = """
DROP TABLE IF EXISTS huishoudens; DROP TABLE IF EXISTS personen; DROP TABLE IF EXISTS zoek;
DROP TABLE IF EXISTS correcties; DROP TABLE IF EXISTS regels;
CREATE TABLE IF NOT EXISTS bewerkingen(id INTEGER PRIMARY KEY, datum TEXT, stap TEXT, versie TEXT,
    beschrijving TEXT, instellingen TEXT, aantallen TEXT, meting TEXT);
CREATE TABLE huishoudens(id INTEGER PRIMARY KEY, sleutel TEXT UNIQUE, boek_id TEXT, pg INT, nr INT,
    regel_van INT, regel_tot INT, begin_reden TEXT, opmaak TEXT, bron TEXT,
    tekst_ocr TEXT, tekst TEXT, aantal_correcties INT);
CREATE TABLE personen(huishouden_id INT, sleutel TEXT, volgnr INT, relatie TEXT, tekst TEXT, leeftijd INT, geb_ca INT);
CREATE TABLE regels(ocr_vorm TEXT PRIMARY KEY, verbeterd TEXT, soort TEXT, aantal INT);
CREATE TABLE correcties(huishouden_id INT, sleutel TEXT, positie_ocr INT, positie INT, ocr TEXT, verbeterd TEXT,
    laag TEXT, regel TEXT, versie TEXT);
CREATE VIRTUAL TABLE zoek USING fts5(tekst, tekst_ocr, huishouden_id UNINDEXED, tokenize='trigram');
"""

def opmaak_van_boek(teksten):
    regels = [l for t in teksten for l in (t or "").splitlines() if len(l.strip()) > 3]
    if not regels:
        return "proza"
    return "lijst" if statistics.median(len(l) for l in regels) < 40 else "proza"

def bewerk(pad, log=print, meting_sweep=None, meting_split=None):
    db = sqlite3.connect(pad)
    db.execute("PRAGMA journal_mode=WAL")
    boeken = {b: j for b, j in db.execute("SELECT boek_id, jaar FROM boeken")}
    paginas = db.execute("SELECT boek_id, pg, ruwe_tekst FROM paginas ORDER BY boek_id, pg").fetchall()
    heeft_lijst = db.execute("SELECT name FROM sqlite_master WHERE name='lijstregels'").fetchone() is not None
    # bestaande (notebook-)splitsing bewaren voor de witregelgrenzen
    oud = collections.defaultdict(list)
    if db.execute("SELECT name FROM sqlite_master WHERE name='huishoudens'").fetchone():
        kol = [r[1] for r in db.execute("PRAGMA table_info(huishoudens)")]
        veld = "tekst_ocr" if "tekst_ocr" in kol else "tekst"
        if "regel_van" in kol:
            for b, pg, rv, rt, reden in db.execute("SELECT boek_id,pg,regel_van,regel_tot,begin_reden FROM huishoudens ORDER BY boek_id,pg,nr"):
                oud[(b, pg)].append(("bereik", rv, rt, reden))
        else:
            for b, pg, t in db.execute(f"SELECT boek_id,pg,{veld} FROM huishoudens ORDER BY boek_id,pg,nr"):
                oud[(b, pg)].append(t)
    if not db.execute("SELECT name FROM sqlite_master WHERE name='witregels'").fetchone():
        # eenmalig: witregelgrenzen uit de notebook-splitsing vastleggen, want die zijn
        # straks niet meer af te leiden (de OCR-posities zijn niet bewaard)
        db.execute("CREATE TABLE witregels(boek_id TEXT, pg INT, na_regel INT)")
        for (b, pg, t) in paginas:
            lst = oud.get((b, pg), [])
            if lst and isinstance(lst[0], str):
                rb = oude_bereiken((t or "").split("\n"), lst)
                for s, e in (rb or [])[:-1]:
                    db.execute("INSERT INTO witregels VALUES (?,?,?)", (b, pg, e))
        db.commit()
    wit = collections.defaultdict(set)
    for b, pg, e in db.execute("SELECT boek_id, pg, na_regel FROM witregels"):
        wit[(b, pg)].add(e)

    per_boek = collections.defaultdict(list)
    for b, pg, t in paginas:
        per_boek[b].append(t)
    opmaak = {b: opmaak_van_boek(ts) for b, ts in per_boek.items()}
    try:
        db.execute("ALTER TABLE boeken ADD COLUMN opmaak TEXT")
    except sqlite3.OperationalError:
        pass
    for b, o in opmaak.items():
        db.execute("UPDATE boeken SET opmaak=? WHERE boek_id=?", (o, b))

    db.executescript(SCHEMA)
    corr = Corrector([(b, t) for b, _, t in paginas])
    reden_tel = collections.Counter(); laag_tel = collections.Counter(); regel_tel = collections.Counter()
    n_hh = 0
    for b, pg, t in paginas:
        lines = (t or "").split("\n")
        lijst_hier = heeft_lijst and db.execute(
            "SELECT 1 FROM lijstregels WHERE boek_id=? AND pg=? LIMIT 1", (b, pg)).fetchone()
        if lijst_hier:
            stukken = lijst_huishoudens(db, b, pg); bron = "kolom-OCR (stap 3)"
        else:
            g = grenzen(lines, wit[(b, pg)], opmaak[b] == "lijst")
            stukken = splits_pagina(lines, g); bron = "OCR notebook v1"
        for nr, (rv, rt, reden, tekst_ocr) in enumerate(stukken, start=1):
            reden_tel[reden.split(" '")[0]] += 1
            sleutel = f"{b}-{pg}-{nr:02d}"
            tekst, wijz = corr.tekst(tekst_ocr)
            cur = db.execute("INSERT INTO huishoudens(sleutel,boek_id,pg,nr,regel_van,regel_tot,begin_reden,"
                             "opmaak,bron,tekst_ocr,tekst,aantal_correcties) VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",
                             (sleutel, b, pg, nr, rv, rt, reden, opmaak[b], bron, tekst_ocr, tekst,
                              sum(1 for w in wijz if w[4] != "rommelteken")))
            hid = cur.lastrowid; n_hh += 1
            db.executemany("INSERT INTO correcties VALUES (?,?,?,?,?,?,?,?,?)",
                           [(hid, sleutel, p, q, o, n, l, r, VERSIE) for p, q, o, n, l, r in wijz])
            for _, _, o, n, l, r in wijz:
                laag_tel[l] += 1
                if r:
                    regel_tel[r] += 1
            db.execute("INSERT INTO zoek(tekst, tekst_ocr, huishouden_id) VALUES (?,?,?)", (tekst, tekst_ocr, hid))
            for i, p in enumerate(ontleed(tekst, boeken.get(b)), start=1):
                db.execute("INSERT INTO personen VALUES (?,?,?,?,?,?,?)",
                           (hid, sleutel, i, p["relatie"], p["tekst"], p["leeftijd"], p["geb_ca"]))
    for v, (n, soort) in corr.regels.items():
        db.execute("INSERT INTO regels VALUES (?,?,?,?)", (v, n, soort, regel_tel.get(v, 0)))
    nu = datetime.datetime.now().isoformat(timespec="seconds")
    db.execute("INSERT INTO bewerkingen(datum,stap,versie,beschrijving,instellingen,aantallen,meting) VALUES (?,?,?,?,?,?,?)",
        (nu, "1 · huishoudens splitsen", VERSIE,
         "Huishoudens bepaald uit de regels van de OCR. Een huishouden eindigt bij een witregel, bij een regel "
         "die duidelijk korter is dan de volle regelbreedte en op een leeftijd eindigt (de typemachine geeft elke "
         "letter even veel ruimte, dus regellengte = breedte), of als de volgende regel begint met een woord dat "
         "een huishouden opent (colonus, vidua, Kötter, incolae, …). Nooit na een regel die op een komma of "
         "afbreekstreepje eindigt. Folio- en letterkoppen worden overgeslagen. Boeken met één persoon per regel "
         "worden in deze stap alleen op witregels gesplitst (zie stap 3).",
         json.dumps({"korte_regel_factor": 0.82, "volle_breedte": "85e percentiel van de regellengte op de pagina"}),
         json.dumps({"huishoudens": n_hh, **reden_tel}, ensure_ascii=False), meting_split))
    db.execute("INSERT INTO bewerkingen(datum,stap,versie,beschrijving,instellingen,aantallen,meting) VALUES (?,?,?,?,?,?,?)",
        (nu, "2 · OCR-correctie", VERSIE,
         "Automatische correctie in drie lagen; de oorspronkelijke OCR blijft altijd bewaard (tekst_ocr). "
         "(1) Naamregels: systematische fouten in het begin van veelvoorkomende namen, vooral door de verhoogd "
         "getypte hoofdletters ('laria' → 'Maria', 'Pernardus' → 'Bernardus', 'enricus' → 'Henricus'); zie tabel "
         "regels. (2) Eén teken: een zeldzaam woord wordt vervangen door een veel vaker voorkomend woord als het "
         "verschil één bekende OCR-verwisseling is; e↔a bewust uitgesloten vanwege echte varianten (Anne/Anna). "
         "(3) Rommeltekens (|, », ^) verwijderd en cijferfouten hersteld (4C → 40, ll → 11). Elke wijziging staat "
         "in de tabel correcties.",
         json.dumps({"naamregels": len(corr.regels), "lexicon": len(corr.lexicon),
                     "min_frequentieverhouding_een_teken": 20}),
         json.dumps(dict(laag_tel), ensure_ascii=False), meting_sweep))
    db.commit()
    log(f"{n_hh} huishoudens; correcties per laag: {dict(laag_tel)}")
    return db

# ---------------------------------------------------------------- lijstboeken (stap 3)
HOOFDROL = re.compile(r"^(?!ehefrau|frau|kinder|kind|kindt|sohn|tochter|vatter|vater|mutter|knecht|knegt|knegte|"
                      r"magd|mohne|bruder|schwester|ux|uxor|conj|fil|filius|filia|filii|prol|proles|lib|liberi|"
                      r"anc|ancilla|serv|servus|servi|fam|famul|famula|soror|frater|pater|mater|avia|avus|nepos|neptis|"
                      r"socrus|socer|gener|nurus|privign|dessen|deren|die|der|et|&|ejus|eius|illorum|prioris|item|ejusdem)", re.I)

def lijst_huishoudens(db, b, pg):
    """Huishoudens uit de kolomregels van stap 3: nieuw huishouden bij een witregel boven de regel
    of bij een rol die geen gezinslid of personeel aanduidt (beroep, Einwohner, colonus, …)."""
    rijen = db.execute("SELECT regelnr, blok, rol, naam, leeftijd, opmerking, witregel_boven FROM lijstregels "
                       "WHERE boek_id=? AND pg=? ORDER BY blok, regelnr", (b, pg)).fetchall()
    uit, huidig, reden, vorig_blok = [], [], "begin van de pagina", None
    def sluit():
        if huidig:
            delen = []
            for r in huidig:
                naam = r[3] or ""
                if r[4]:                                  # losse cijfers achter de naam zijn ruis (verhoogde letter)
                    naam = re.sub(r"(\s+[\d|:;.,]{1,3})+$", "", naam)
                stuk = " ".join(x for x in (r[2], naam, r[4], r[5]) if x)
                delen.append(stuk)
            uit.append((huidig[0][0], huidig[-1][0], reden, ", ".join(delen)))
    for r in rijen:
        regelnr, blok, rol, naam, lft, opm, wit = r
        nieuw = None
        if blok != vorig_blok and huidig:
            nieuw = "volgende kolom op de pagina"
        elif wit and huidig:
            nieuw = "witregel"
        elif rol and HOOFDROL.match(rol.strip(" .:")) and huidig and naam:
            nieuw = f"rol '{rol.strip(' .:')}' opent een huishouden"
        if nieuw:
            sluit(); huidig, reden = [], nieuw
        huidig.append(r); vorig_blok = blok
    sluit()
    return uit

In [ ]:
%%writefile /content/lijst_ocr.py
# lijst_ocr.py – stap 3: kolomgewijze OCR voor boeken met één persoon per regel
import os, re
from concurrent.futures import ThreadPoolExecutor
# Tesseract mag per aanroep maar één kern gebruiken; we draaien zelf meerdere kolommen tegelijk.
# Dat verandert niets aan de uitkomst, alleen aan de snelheid.
os.environ.setdefault("OMP_THREAD_LIMIT", "1")
import numpy as np
import cv2
import pytesseract

LIJST_VERSIE = "2026-11-04.1"   # snellere code, uitkomst gelijk aan de eerste versie
WERKERS = max(1, min(4, os.cpu_count() or 1))
LEEFTIJD = re.compile(r"^(\d{1,3}|\d?\s?1/[24]|[13]/4|1/2)[.,;:]?$")

def _gamma(g):
    return np.clip(((g / 255.0) ** 2.0) * 255, 0, 255).astype(np.uint8)

def _inkt(g):
    thr = min(170, cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[0])
    return (g < thr).astype(np.uint8)

def _inhoud_x(ink):
    """x-bereik van de bladspiegel: zwarte randen (rug, scanrand) eraf."""
    h, w = ink.shape
    frac = ink.sum(axis=0) / h
    ok = np.where(frac < 0.35)[0]
    if len(ok) == 0:
        return 0, w
    x0, x1 = ok[0], ok[-1]
    # randstroken met veel inkt aan de buitenkant overslaan
    while x0 < w // 4 and frac[x0] > 0.15:
        x0 += 1
    while x1 > 3 * w // 4 and frac[x1] > 0.15:
        x1 -= 1
    return int(x0) + 5, int(x1) - 5

def _segmenten(ink, x0, x1, min_gat):
    """Kolommen: x-stroken met tekst, gescheiden door verticale witruimte van minstens min_gat px."""
    h = ink.shape[0]
    prof = ink[int(0.04 * h):int(0.97 * h), x0:x1].sum(axis=0)
    prof = np.convolve(prof, np.ones(5) / 5, mode="same")
    leeg = prof <= max(2, 0.08 * np.percentile(prof, 95))
    seg, start, gat = [], None, 0
    for i, l in enumerate(leeg):
        if not l:
            if start is None:
                start = i
            gat = 0
        else:
            if start is not None:
                gat += 1
                if gat >= min_gat:
                    seg.append([start, i - gat + 1]); start = None; gat = 0
    if start is not None:
        seg.append([start, len(leeg)])
    seg = [[a + x0, b + x0] for a, b in seg if b - a > 12]
    return seg

def _blokken(seg, x0, x1):
    """Twee blokken naast elkaar (dubbele pagina, of twee huishoudlijsten naast elkaar) als er
    rond het midden een brede goot zit met aan beide kanten veel tekst."""
    w = x1 - x0
    for i in range(len(seg) - 1):
        gat = seg[i + 1][0] - seg[i][1]
        midden = (seg[i][1] + seg[i + 1][0]) / 2
        if gat > 0.05 * w and 0.33 * w < midden - x0 < 0.67 * w:
            links, rechts = seg[:i + 1], seg[i + 1:]
            if links[-1][1] - links[0][0] > 0.25 * w and rechts[-1][1] - rechts[0][0] > 0.25 * w:
                return [links, rechts]
    return [seg]

def _ocr_strook(g, a, b, psm=6):
    crop = _gamma(g[:, max(0, a - 6):b + 6])
    d = pytesseract.image_to_data(crop, lang="lat+deu", config=f"--psm {psm}",
                                  output_type=pytesseract.Output.DICT)
    woorden = []
    for i, t in enumerate(d["text"]):
        if t.strip() and float(d["conf"][i]) >= 0:
            woorden.append({"t": t, "y": d["top"][i] + d["height"][i] / 2, "top": d["top"][i],
                            "bot": d["top"][i] + d["height"][i], "x": d["left"][i] + a, "h": d["height"][i]})
    return woorden

def lijst_ocr(beeld_bytes):
    """Geeft rijen: dict(blok, regelnr, kolommen=[tekst per kolom], top, bot) en meta."""
    g = cv2.imdecode(np.frombuffer(beeld_bytes, np.uint8), cv2.IMREAD_GRAYSCALE)
    ink = _inkt(g)
    x0, x1 = _inhoud_x(ink)
    min_gat = max(18, int(0.012 * (x1 - x0)))
    seg = _segmenten(ink, x0, x1, min_gat)
    rijen_uit = []
    with ThreadPoolExecutor(max_workers=WERKERS) as pool:     # kolommen tegelijk lezen
        res = list(pool.map(lambda s_: _ocr_strook(g, s_[0], s_[1]), seg))
    kolwoorden_seg = {tuple(s_): r for s_, r in zip(seg, res)}
    def numeriek(s_):
        ws = kolwoorden_seg[tuple(s_)]
        return bool(ws) and sum(1 for w in ws if LEEFTIJD.match(w["t"].strip(".,"))) / len(ws) > 0.5
    def tekst(s_):
        return sum(len(w["t"]) for w in kolwoorden_seg[tuple(s_)] if not LEEFTIJD.match(w["t"].strip(".,")))
    blokken = _blokken(seg, x0, x1)
    if len(blokken) == 2:
        maxt = max(tekst(s_) for s_ in seg) or 1
        if not all(any(not numeriek(s_) and tekst(s_) > 0.3 * maxt for s_ in b) for b in blokken):
            blokken = [seg]                       # rechts alleen leeftijden/opmerkingen: één blok
    for bi, blok in enumerate(blokken):
        kolwoorden = [kolwoorden_seg[tuple(s_)] for s_ in blok]
        alle = [w for kw in kolwoorden for w in kw]
        if not alle:
            continue
        hmed = float(np.median([w["h"] for w in alle])) or 20
        # regels vormen over alle kolommen heen, op verticale positie
        alle.sort(key=lambda w: w["y"])
        regels = []
        for w in alle:
            if regels and abs(w["y"] - regels[-1]["y"]) < 0.6 * hmed:
                r = regels[-1]; r["w"].append(w); r["y"] = np.mean([x["y"] for x in r["w"]])
                r["top"] = min(r["top"], w["top"]); r["bot"] = max(r["bot"], w["bot"])
            else:
                regels.append({"y": w["y"], "w": [w], "top": w["top"], "bot": w["bot"]})
        afstanden = [b["top"] - a["bot"] for a, b in zip(regels, regels[1:])]
        normaal = float(np.median(afstanden)) if afstanden else hmed * 0.5
        for ri, r in enumerate(regels):
            kols = []
            for (a, b) in blok:
                kols.append(" ".join(w["t"] for w in sorted(r["w"], key=lambda w: w["x"]) if a - 8 <= w["x"] < b + 8))
            wit = ri > 0 and (r["top"] - regels[ri - 1]["bot"]) > normaal + 0.7 * hmed
            rijen_uit.append({"blok": bi, "regelnr": ri, "kolommen": kols, "top": int(r["top"]),
                              "bot": int(r["bot"]), "witregel_boven": bool(wit)})
    return rijen_uit, {"kolommen": [[a, b] for a, b in seg], "blokken": len(blokken)}

def ken_kolommen_toe(rijen):
    """Per blok: welke kolom is rol / naam / leeftijd / opmerking (op basis van de inhoud)."""
    per_blok = {}
    for r in rijen:
        per_blok.setdefault(r["blok"], []).append(r)
    uit = []
    for b, rs in per_blok.items():
        n = max(len(r["kolommen"]) for r in rs)
        tekens = [sum(len(r["kolommen"][k]) for r in rs if k < len(r["kolommen"])) for k in range(n)]
        getal = [sum(1 for r in rs if k < len(r["kolommen"]) and r["kolommen"][k] and
                     LEEFTIJD.match(r["kolommen"][k].split()[0].strip(".,")) ) for k in range(n)]
        gevuld = [sum(1 for r in rs if k < len(r["kolommen"]) and r["kolommen"][k].strip()) for k in range(n)]
        naam = int(np.argmax(tekens))
        lft = [k for k in range(naam + 1, n) if gevuld[k] and getal[k] / gevuld[k] > 0.5]
        lft = lft[0] if lft else None
        for r in rs:
            k = r["kolommen"] + [""] * (n - len(r["kolommen"]))
            rol = " ".join(x for x in k[:naam] if x).strip()
            nm = k[naam].strip()
            leeftijd = k[lft].strip() if lft is not None else ""
            rest = " ".join(x for i, x in enumerate(k) if i > naam and i != lft and x).strip()
            if lft is None or not leeftijd:                    # leeftijd soms in de naamkolom
                m = re.search(r"\s(\d{1,3}|1/[24]|3/4)[.,]?$", nm)
                if m:
                    leeftijd, nm = m.group(1), nm[:m.start()].strip()
            # rol in de naamkolom ('filiae: Margaretha', 'ejus uxor: Aleidis')
            m = re.match(r"^((?:[A-Za-zÄÖÜäöüß()?.]+\s){0,2}[A-Za-zÄÖÜäöüß()?.]+)\s?[:;]\s*(.*)$", nm)
            if m and len(m.group(1)) <= 24 and not rol:
                rol, nm = m.group(1).strip(), m.group(2).strip()
            uit.append({**r, "rol": rol, "naam": nm, "leeftijd": leeftijd, "opmerking": rest})
    return uit

### 5c. Stap 3: lijstboeken per kolom OCR'en (achtergrond)

In [ ]:
_nodig = [n for n in ("open_db", "lees_boek", "haal_beeld", "log", "bewaar_op_drive", "DB_LOKAAL") if n not in globals()]
if _nodig:
    raise RuntimeError("Draai eerst cel 1 tot en met 3: installeren, opslag (Drive) en de drie cellen onder "
                       "'De machinerie'. Na een herstart van Colab moet dat opnieuw. Ontbreekt nu: " + ", ".join(_nodig))
import sys, importlib, threading, json, datetime, queue
sys.path.insert(0, "/content")
import bewerk, lijst_ocr
importlib.reload(bewerk); importlib.reload(lijst_ocr)

def _stap3(stop):
    conn = open_db(DB_LOKAAL)
    conn.execute("""CREATE TABLE IF NOT EXISTS lijstregels(boek_id TEXT, pg INT, blok INT, regelnr INT,
        rol TEXT, naam TEXT, leeftijd TEXT, opmerking TEXT, witregel_boven INT, top INT, bot INT, versie TEXT)""")
    # welke boeken zijn lijstboeken? (zelfde regel als in stap 1)
    per_boek = {}
    for b, t in conn.execute("SELECT boek_id, ruwe_tekst FROM paginas"):
        per_boek.setdefault(b, []).append(t)
    lijst = sorted(b for b, ts in per_boek.items() if bewerk.opmaak_van_boek(ts) == "lijst")
    log(f"Stap 3: {len(lijst)} lijstboeken")
    n_pag = 0
    try:
        for b in lijst:
            if stop.is_set():
                break
            url = conn.execute("SELECT basis_url FROM boeken WHERE boek_id=?", (b,)).fetchone()[0]
            klaar = {r[0] for r in conn.execute("SELECT DISTINCT pg FROM lijstregels WHERE boek_id=?", (b,))}
            sessie = requests.Session()
            boek = lees_boek(url, sessie)
            todo = [(i + 1, pad, label) for i, (pad, label) in enumerate(boek["paginas"]) if i + 1 not in klaar]
            # downloaden en lezen overlappen: een aparte draad haalt alvast de volgende scan op,
            # met nog steeds minstens PAUZE_SEC tussen twee verzoeken aan Matricula
            wachtrij = queue.Queue(maxsize=2)
            def ophalen():
                for pg, pad, label in todo:
                    if stop.is_set():
                        break
                    beeld_ = haal_beeld(pad, boek["csrf"], sessie)
                    time.sleep(PAUZE_SEC)               # zelfde pauze als altijd
                    wachtrij.put((pg, label, beeld_))
                wachtrij.put(None)
            threading.Thread(target=ophalen, daemon=True).start()
            while not stop.is_set():
                item = wachtrij.get()
                if item is None:
                    break
                pg, label, beeld = item
                if beeld is None:
                    log(f" · {b} {label} – overgeslagen"); continue
                rijen, meta = lijst_ocr.lijst_ocr(beeld)
                del beeld
                rs = lijst_ocr.ken_kolommen_toe(rijen)
                conn.executemany("INSERT INTO lijstregels VALUES (?,?,?,?,?,?,?,?,?,?,?,?)",
                    [(b, pg, r["blok"], r["regelnr"], r["rol"], r["naam"], r["leeftijd"], r["opmerking"],
                      int(r["witregel_boven"]), r["top"], r["bot"], lijst_ocr.LIJST_VERSIE) for r in rs])
                conn.commit(); n_pag += 1
                log(f" · {b} {label} (pg={pg}) – {len(rs)} regels, {meta['blokken']} blok(ken), {len(meta['kolommen'])} kolommen")
                bewaar_op_drive(conn, stil=True)
    except Exception as e:
        log(f"FOUT in stap 3: {e!r}")
    finally:
        totaal = conn.execute("SELECT COUNT(DISTINCT boek_id||'-'||pg) FROM lijstregels").fetchone()[0]
        conn.execute("CREATE TABLE IF NOT EXISTS bewerkingen(id INTEGER PRIMARY KEY, datum TEXT, stap TEXT, versie TEXT, "
                     "beschrijving TEXT, instellingen TEXT, aantallen TEXT, meting TEXT)")
        conn.execute("DELETE FROM bewerkingen WHERE stap LIKE '3 %'")
        conn.execute("INSERT INTO bewerkingen(datum,stap,versie,beschrijving,instellingen,aantallen,meting) VALUES (?,?,?,?,?,?,?)",
            (datetime.datetime.now().isoformat(timespec="seconds"), "3 · kolom-OCR van lijstboeken", lijst_ocr.LIJST_VERSIE,
             "Boeken met één persoon per regel (mediane regellengte onder 40 tekens) opnieuw geOCR'd, per kolom: "
             "de kolommen worden gevonden aan verticale witruimte op de pagina, elke kolom wordt apart gelezen en de "
             "regels worden weer samengevoegd op hun hoogte op de pagina. De kolom met de meeste tekst is de naam, "
             "links daarvan de rol, de eerste kolom rechts met vooral getallen de leeftijd, de rest opmerkingen. "
             "Twee lijsten naast elkaar (of een dubbele pagina) worden als twee blokken gelezen. Een huishouden begint "
             "bij een witregel of bij een rol die geen gezinslid of personeel is (beroep, Einwohner, colonus, …). "
             "De oorspronkelijke OCR van deze pagina's blijft in paginas.ruwe_tekst staan.",
             json.dumps({"lijstboeken": len(lijst), "pauze_sec": PAUZE_SEC}),
             json.dumps({"paginas_kolom_ocr": totaal}),
             "Getest op 8 steekproefpagina's van verschillende lijstboeken; kolommen juist herkend op 7 daarvan "
             "(Lüdinghausen pg 4 heeft een afwijkende opbouw). Op een met de hand overgetypte pagina van Sendenhorst "
             "(AS_019): 150 van 183 woorden in naam- en leeftijdskolom juist, en de rolkolom (Ehefrau, Kinder, …) "
             "grotendeels leesbaar, waar die bij gewone OCR onbruikbaar was."))
        conn.commit(); bewaar_op_drive(conn, stil=True); conn.close()
        log(f"=== STAP 3 {'GESTOPT' if stop.is_set() else 'KLAAR'} ({n_pag} nieuwe pagina's) ===")

if "werker" in globals() and werker.is_alive():
    print("Er loopt al een achtergrondtaak. Stop die eerst (cel 4c) of wacht tot hij klaar is.")
else:
    STOP = threading.Event()
    werker = threading.Thread(target=_stap3, args=(STOP,), daemon=True)
    werker.start()
    print("Stap 3 gestart op de achtergrond. Voortgang: cel 4b (laatste meldingen). Stoppen: cel 4c.")

### 5d. (optioneel) Stap 3 controleren op één pagina
Toont voor één scan van een lijstboek hoe de kolommen zijn gelezen. Pas `BOEK` en `PG` aan.

In [ ]:
BOEK, PG = "SAA_SenM", 20
rs = db.execute("SELECT blok, witregel_boven, rol, naam, leeftijd, opmerking FROM lijstregels "
                "WHERE boek_id=? AND pg=? ORDER BY blok, regelnr", (BOEK, PG)).fetchall()
for blok, wit, rol, naam, lft, opm in rs:
    print(f"{blok} {'—' if wit else ' '} {rol[:20]:20} | {naam[:45]:45} | {lft:6} | {opm[:25]}")

### 5e. Stap 1 en 2: huishoudens splitsen en OCR-correctie
Bouwt alles opnieuw op uit de ruwe OCR (en uit de kolom-OCR van stap 3, voor zover die er is). Draai dit na stap 3, en opnieuw als je later iets verandert. Duurt ongeveer een halve minuut.

In [ ]:
_nodig = [n for n in ("open_db", "lees_boek", "haal_beeld", "log", "bewaar_op_drive", "DB_LOKAAL") if n not in globals()]
if _nodig:
    raise RuntimeError("Draai eerst cel 1 tot en met 3: installeren, opslag (Drive) en de drie cellen onder "
                       "'De machinerie'. Na een herstart van Colab moet dat opnieuw. Ontbreekt nu: " + ", ".join(_nodig))
import sys, importlib
sys.path.insert(0, "/content")
import bewerk; importlib.reload(bewerk)
if "werker" in globals() and werker.is_alive():
    print("Wacht tot de achtergrondtaak klaar is (cel 4b), of stop hem met cel 4c.")
else:
    db.close()
    MET_SPLIT = ("Gecontroleerd op twee willekeurige pagina's zonder witregels: Warendorf (SAA_WadM pg 13) alle "
                 "grenzen in het gecontroleerde deel juist; Greven (SAA_GreM pg 9) op enkele na juist. Bekende "
                 "zwakte: een huishouden waarvan de laatste regel toevallig de volle breedte heeft, wordt niet "
                 "gesplitst van het volgende.")
    MET_SWEEP = ("Op twee met de hand overgetypte pagina's: Sendenhorst AS_019 van 80% naar 87% juiste woorden, Ahaus "
                 "AS_001 van 92% naar 94%. In een willekeurige controle van 70 naamregelcorrecties vrijwel alle juist; "
                 "van 70 één-teken-correcties ongeveer 1 op 7 twijfelachtig of fout (bijv. 'GroBe' → 'Grote' waar "
                 "'Große' bedoeld was). Spelling van de typist zelf kan worden 'verbeterd' ('Hennricus' → 'Henricus').")
    nieuw = bewerk.bewerk(DB_LOKAAL, log=print, meting_sweep=MET_SWEEP, meting_split=MET_SPLIT)
    nieuw.close()
    db = open_db(DB_LOKAAL)
    bewaar_op_drive(db)
    for stap, versie, aantallen in db.execute("SELECT stap, versie, aantallen FROM bewerkingen ORDER BY id DESC LIMIT 3"):
        print(f"{stap} ({versie}): {aantallen}")

### 5f. Wat is er veranderd?
De meest toegepaste naamregels en een willekeurige greep uit de overige correcties.

In [ ]:
print("Meest toegepaste naamregels:")
for v, n, soort, a in db.execute("SELECT ocr_vorm, verbeterd, soort, aantal FROM regels ORDER BY aantal DESC LIMIT 25"):
    print(f"  {v:14} → {n:12} {a:6}×  ({soort})")
print("\nWillekeurige één-teken-correcties:")
for o, n, s in db.execute("SELECT ocr, verbeterd, sleutel FROM correcties WHERE laag='één teken' ORDER BY random() LIMIT 25"):
    print(f"  {o:16} → {n:16} {s}")

## 6. Zoeken
- **Fuzzy** (standaard): vergelijkt op klank én spelling, dus *Büning* vindt ook *Büninck*, *Buninck* en *Brüning*. Verlaag de drempel voor meer (ook foute) treffers.
- **Exact**: zoekt letterlijk, ook op deelwoorden (`merinck` vindt Almerinck en Ahlmerinck). Minimaal 3 tekens.
- **Geboren ca.**: alleen huishoudens met minstens één gezinslid wiens geschatte geboortejaar in die periode valt (0 = geen grens).

De link opent de scan op Matricula. Je kunt zoeken terwijl cel 4 op de achtergrond doorloopt; elke nieuwe zoekopdracht neemt de inmiddels verwerkte pagina's mee.

In [ ]:
import ipywidgets as W
from IPython.display import display, HTML

veld = W.Text(placeholder="bijv. Büninck  of  Gerhardus Brinck", description="Zoek:",
              continuous_update=False, layout=W.Layout(width="420px"))
modus = W.ToggleButtons(options=[("Fuzzy", True), ("Exact", False)], value=True)
drempel = W.IntSlider(value=85, min=60, max=100, step=1, description="Drempel")
geb_van = W.IntText(value=0, description="Geb. vanaf", layout=W.Layout(width="180px"))
geb_tot = W.IntText(value=0, description="t/m", layout=W.Layout(width="140px"))
knop = W.Button(description="Zoeken", button_style="primary")
uitvoer = W.Output()

def zoeken(_=None):
    with uitvoer:
        uitvoer.clear_output()
        res = zoek(db, veld.value, fuzzy=modus.value, drempel=drempel.value,
                   geb_van=geb_van.value or None, geb_tot=geb_tot.value or None)
        n_pag, n_boek = db.execute("SELECT COUNT(*), COUNT(DISTINCT boek_id) FROM paginas").fetchone()
        loopt = "werker" in globals() and werker.is_alive()
        display(HTML(f"<p style='color:gray'>Doorzocht: {n_pag} pagina's uit {n_boek} boek(en)"
                     + (" – verwerking loopt nog, dus later kunnen er meer treffers zijn" if loopt else "")
                     + "</p>" + toon(db, res, veld.value)))

knop.on_click(zoeken)
veld.observe(zoeken, names="value")
display(W.VBox([W.HBox([veld, knop]), W.HBox([modus, drempel]), W.HBox([geb_van, geb_tot]), uitvoer]))

## 7. Exporteren naar Excel
Tabbladen *huishoudens* (verbeterde tekst én oorspronkelijke OCR, met link), *personen* (ruwe structuur), en — na stap 1 en 2 — *correcties*, *regels* en *bewerkingen*. De database zelf kun je ook downloaden.

In [ ]:
import pandas as pd
from google.colab import files

bewerkt = "tekst_ocr" in [r[1] for r in db.execute("PRAGMA table_info(huishoudens)")]
pad = "/content/matricula_ocr.xlsx"
with pd.ExcelWriter(pad) as xw:
    if bewerkt:
        pd.read_sql("""SELECT h.sleutel, h.boek_id AS boek, p.label AS scan, h.pg,
                          b.basis_url || '?pg=' || h.pg AS link, h.begin_reden, h.tekst, h.tekst_ocr, h.aantal_correcties
                   FROM huishoudens h JOIN paginas p USING(boek_id, pg) JOIN boeken b USING(boek_id)
                   ORDER BY h.boek_id, h.pg, h.nr""", db).to_excel(xw, sheet_name="huishoudens", index=False)
        pd.read_sql("SELECT sleutel, volgnr, relatie, tekst, leeftijd, geb_ca FROM personen ORDER BY huishouden_id, volgnr",
                    db).to_excel(xw, sheet_name="personen", index=False)
        pd.read_sql("SELECT sleutel, positie_ocr, positie, ocr, verbeterd, laag, regel, versie FROM correcties WHERE laag != 'rommelteken'",
                    db).to_excel(xw, sheet_name="correcties", index=False)
        pd.read_sql("SELECT * FROM regels ORDER BY aantal DESC", db).to_excel(xw, sheet_name="regels", index=False)
        pd.read_sql("SELECT * FROM bewerkingen ORDER BY id", db).to_excel(xw, sheet_name="bewerkingen", index=False)
    else:
        pd.read_sql("""SELECT h.id AS huishouden, h.boek_id AS boek, p.label AS scan, h.pg,
                          b.basis_url || '?pg=' || h.pg AS link, h.tekst AS ocr_tekst
                   FROM huishoudens h JOIN paginas p USING(boek_id, pg) JOIN boeken b USING(boek_id)
                   ORDER BY h.boek_id, h.pg, h.nr""", db).to_excel(xw, sheet_name="huishoudens", index=False)
        pd.read_sql("SELECT * FROM personen", db).to_excel(xw, sheet_name="personen", index=False)
print("Excel gemaakt:", pad)
files.download(pad)
# files.download(DB_LOKAAL)   # haal het #-teken weg om ook de database te downloaden